# Run the Complete Checkpoint-Based Reproduction

This notebook reproduces the reported evaluation without retraining either
model.

It performs the following workflow:

```text
clone the public source-data repository
    → locate the FA, FCA, and FCB source folders
    → optionally regenerate the converted graph annotations
    → resize and place the corresponding images at 1024 × 1024
    → load the two supplied frozen checkpoints
    → run two-pass inference and evaluation
```

The publication notebooks retain blank path fields. This runner creates
temporary working copies, inserts all resolved paths automatically, and executes
those copies in isolated kernels.

By default, the supplied converted annotations are used. Set
`REGENERATE_ANNOTATIONS = True` only when the annotation conversion itself
should also be reproduced.


In [ ]:
# ============================================================
# USER SETTINGS
# ============================================================

# Leave blank when this notebook remains inside the repository's notebooks/
# folder or Jupyter is launched from the repository root.
PROJECT_ROOT = ""

SOURCE_DATASET_GITHUB_URL = (
    "https://github.com/bernhardschaefer/handwritten-diagram-datasets.git"
)

# Leave blank to use the current default branch of the source repository.
SOURCE_DATASET_COMMIT = ""

# False uses the converted JSON annotations supplied with this release.
# True runs Notebook 00 before image reconstruction.
REGENERATE_ANNOTATIONS = False

UPDATE_EXISTING_DATASET_CLONE = False
FORCE_REDOWNLOAD_DATASET = False

# Normally install requirements once outside the notebook and leave this False.
INSTALL_REQUIREMENTS = False

KERNEL_NAME = "python3"
NOTEBOOK_CELL_TIMEOUT_SECONDS = -1
STOP_ON_FIRST_ERROR = True


## 1. Imports and repository discovery

When `PROJECT_ROOT` is blank, the runner searches the current directory and its
parents for the repository root.


In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import shutil
import subprocess
import sys
import time
import traceback
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import nbformat
from nbclient import NotebookClient


def resolve_project_root(configured_root: str) -> Path:
    configured_root = str(configured_root).strip()

    if configured_root:
        candidate = Path(configured_root).expanduser().resolve()
        if not candidate.exists():
            raise FileNotFoundError(f"PROJECT_ROOT does not exist: {candidate}")
        return candidate

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate

    if current.name == "notebooks" and (current.parent / "README.md").is_file():
        return current.parent

    raise RuntimeError(
        "Could not determine the repository root automatically. "
        "Fill PROJECT_ROOT in the USER SETTINGS cell."
    )


PROJECT_ROOT_PATH = resolve_project_root(PROJECT_ROOT)

NOTEBOOKS_DIR = PROJECT_ROOT_PATH / "notebooks"
TRAINED_MODELS_DIR = PROJECT_ROOT_PATH / "trained_models"
CONVERTED_DATASET_ROOT = PROJECT_ROOT_PATH / "converted_annotations"
SAMPLE_OUTPUTS_ROOT = PROJECT_ROOT_PATH / "sample_outputs"

EXTERNAL_DATA_ROOT = PROJECT_ROOT_PATH / "external_data"
SOURCE_CLONE_DIR = EXTERNAL_DATA_ROOT / "handwritten-diagram-datasets"

WORKSPACE_ROOT = PROJECT_ROOT_PATH / "workspace"
PREPARED_NOTEBOOKS_DIR = WORKSPACE_ROOT / "prepared_notebooks"
EXECUTED_NOTEBOOKS_DIR = WORKSPACE_ROOT / "executed_notebooks"
FINAL_OUTPUT_ROOT = SAMPLE_OUTPUTS_ROOT / "final_run"

for directory in (
    EXTERNAL_DATA_ROOT,
    WORKSPACE_ROOT,
    PREPARED_NOTEBOOKS_DIR,
    EXECUTED_NOTEBOOKS_DIR,
    FINAL_OUTPUT_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT_PATH}")


## 2. Optional environment installation

In [ ]:
def run_command(
    command: Sequence[str],
    cwd: Optional[Path] = None,
    capture_output: bool = False,
) -> subprocess.CompletedProcess:
    printable = " ".join(str(part) for part in command)
    print(f"$ {printable}")

    return subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd) if cwd is not None else None,
        check=True,
        text=True,
        capture_output=capture_output,
    )


if INSTALL_REQUIREMENTS:
    requirements_path = PROJECT_ROOT_PATH / "requirements.txt"
    if not requirements_path.is_file():
        raise FileNotFoundError(f"Missing requirements file: {requirements_path}")

    run_command(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-r",
            str(requirements_path),
        ]
    )


## 3. Download and locate the public source datasets

The exact Git commit used for reproduction is recorded in the final manifest.


In [ ]:
REQUIRED_FAMILIES = ("fa", "fca", "fcb")


def ensure_git_available() -> None:
    try:
        run_command(["git", "--version"], capture_output=True)
    except Exception as exc:
        raise RuntimeError(
            "Git is required to download the public source dataset repository."
        ) from exc


def clone_or_update_source_repository() -> Path:
    ensure_git_available()
    EXTERNAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

    if FORCE_REDOWNLOAD_DATASET and SOURCE_CLONE_DIR.exists():
        print(f"Removing existing source clone: {SOURCE_CLONE_DIR}")
        shutil.rmtree(SOURCE_CLONE_DIR)

    if not SOURCE_CLONE_DIR.exists():
        run_command(
            [
                "git",
                "clone",
                "--depth",
                "1",
                SOURCE_DATASET_GITHUB_URL,
                str(SOURCE_CLONE_DIR),
            ]
        )
    elif not (SOURCE_CLONE_DIR / ".git").exists():
        raise RuntimeError(
            f"The source clone path exists but is not a Git repository: "
            f"{SOURCE_CLONE_DIR}"
        )
    elif UPDATE_EXISTING_DATASET_CLONE:
        run_command(["git", "fetch", "--all", "--tags"], cwd=SOURCE_CLONE_DIR)
        run_command(["git", "pull", "--ff-only"], cwd=SOURCE_CLONE_DIR)
    else:
        print(f"Using existing source clone: {SOURCE_CLONE_DIR}")

    if str(SOURCE_DATASET_COMMIT).strip():
        run_command(
            ["git", "fetch", "--depth", "1", "origin", SOURCE_DATASET_COMMIT],
            cwd=SOURCE_CLONE_DIR,
        )
        run_command(
            ["git", "checkout", "--detach", SOURCE_DATASET_COMMIT],
            cwd=SOURCE_CLONE_DIR,
        )

    return SOURCE_CLONE_DIR


def is_dataset_root(path: Path) -> bool:
    for family in REQUIRED_FAMILIES:
        family_dir = path / family
        if not family_dir.is_dir():
            return False
        if not any(family_dir.glob("*.json")):
            return False
        if not any(child.is_dir() for child in family_dir.iterdir()):
            return False
    return True


def find_source_dataset_root(clone_root: Path) -> Path:
    for candidate in (clone_root / "datasets", clone_root):
        if candidate.exists() and is_dataset_root(candidate):
            return candidate.resolve()

    candidates: List[Path] = []
    for path in clone_root.rglob("*"):
        if not path.is_dir():
            continue
        try:
            depth = len(path.relative_to(clone_root).parts)
        except ValueError:
            continue
        if depth <= 4 and is_dataset_root(path):
            candidates.append(path.resolve())

    unique_candidates = sorted(set(candidates))
    if len(unique_candidates) == 1:
        return unique_candidates[0]
    if len(unique_candidates) > 1:
        formatted = "\n".join(f"  - {path}" for path in unique_candidates)
        raise RuntimeError(
            "More than one possible source dataset root was found:\n"
            f"{formatted}"
        )

    raise RuntimeError(
        "Could not locate a folder containing the required fa/, fca/, and "
        "fcb/ source datasets."
    )


def current_git_commit(repository: Path) -> str:
    result = run_command(
        ["git", "rev-parse", "HEAD"],
        cwd=repository,
        capture_output=True,
    )
    return result.stdout.strip()


SOURCE_REPOSITORY_ROOT = clone_or_update_source_repository()
SOURCE_DATASETS_ROOT = find_source_dataset_root(SOURCE_REPOSITORY_ROOT)
SOURCE_COMMIT_USED = current_git_commit(SOURCE_REPOSITORY_ROOT)

print(f"Source repository : {SOURCE_REPOSITORY_ROOT}")
print(f"Source dataset root: {SOURCE_DATASETS_ROOT}")
print(f"Source commit     : {SOURCE_COMMIT_USED}")


## 4. Validate the supplied reproduction artifacts

In [ ]:
NOTEBOOK_FILES = {
    "annotation_conversion": "00_Convert_Original_Annotations.ipynb",
    "image_resizing": "01_Resize_Dataset_Images_to_1024.ipynb",
    "final_inference": "02_Run_Two_Pass_Inference_and_Evaluation.ipynb",
}

for filename in NOTEBOOK_FILES.values():
    path = NOTEBOOKS_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Missing reproduction notebook: {path}")


SUPPLIED_GRAPH_CHECKPOINT = TRAINED_MODELS_DIR / "fcinkml_best.pt"
SUPPLIED_NODE_CHECKPOINT = TRAINED_MODELS_DIR / "resnet_34.pth"

missing_checkpoints = [
    path
    for path in (SUPPLIED_GRAPH_CHECKPOINT, SUPPLIED_NODE_CHECKPOINT)
    if not path.is_file()
]
if missing_checkpoints:
    formatted = "\n".join(f"  - {path}" for path in missing_checkpoints)
    raise FileNotFoundError(
        "The frozen checkpoints required for reproduction are missing:\n"
        f"{formatted}"
    )


def converted_annotation_files(root: Path) -> List[Path]:
    return sorted(root.glob("*/*/json/*.json"))


if not REGENERATE_ANNOTATIONS:
    supplied_json = converted_annotation_files(CONVERTED_DATASET_ROOT)
    if not supplied_json:
        raise FileNotFoundError(
            "No converted per-image annotations were found under "
            f"{CONVERTED_DATASET_ROOT}. Either supply them or set "
            "REGENERATE_ANNOTATIONS = True."
        )
    print(f"Supplied converted annotations: {len(supplied_json)}")

print("Frozen checkpoints found.")


## 5. Create path-filled working copies

The original notebooks remain unchanged. Only the copies under
`workspace/prepared_notebooks/` receive resolved paths.


In [ ]:
def assignment_literal(value: Any) -> str:
    if isinstance(value, Path):
        value = str(value)
    return repr(value)


def replace_first_assignment(
    source: str,
    variable_name: str,
    value: Any,
) -> Tuple[str, bool]:
    pattern = re.compile(
        rf"(?m)^(?P<indent>[ \t]*){re.escape(variable_name)}\s*=.*$"
    )

    def replacement(match: re.Match) -> str:
        return (
            f"{match.group('indent')}{variable_name} = "
            f"{assignment_literal(value)}"
        )

    updated, count = pattern.subn(replacement, source, count=1)
    return updated, count == 1


def prepare_notebook(
    source_path: Path,
    destination_path: Path,
    assignments: Dict[str, Any],
) -> Path:
    notebook = nbformat.read(source_path, as_version=4)
    unresolved = set(assignments)

    for cell in notebook.cells:
        if cell.cell_type != "code":
            continue

        source = cell.source
        for variable_name in list(unresolved):
            source, replaced = replace_first_assignment(
                source,
                variable_name,
                assignments[variable_name],
            )
            if replaced:
                unresolved.remove(variable_name)

        cell.source = source
        cell.execution_count = None
        cell.outputs = []

    if unresolved:
        raise RuntimeError(
            f"Could not patch variables {sorted(unresolved)} in "
            f"{source_path.name}"
        )

    notebook.cells.insert(
        1,
        nbformat.v4.new_markdown_cell(
            "## Automatically prepared reproduction copy\n\n"
            "This copy was generated by "
            "`03_Run_Complete_Reproduction.ipynb`. The distributed "
            "publication notebook was not modified."
        ),
    )

    destination_path.parent.mkdir(parents=True, exist_ok=True)
    nbformat.write(notebook, destination_path)
    return destination_path


stage_assignments: Dict[str, Dict[str, Any]] = {
    "annotation_conversion": {
        "SOURCE_DATASETS_ROOT": SOURCE_DATASETS_ROOT,
        "OUTPUT_ANNOTATIONS_ROOT": CONVERTED_DATASET_ROOT,
    },
    "image_resizing": {
        "ORIGINAL_DATASET_ROOT": SOURCE_DATASETS_ROOT,
        "CONVERTED_DATASET_ROOT": CONVERTED_DATASET_ROOT,
    },
    "final_inference": {
        "DATASET_ROOT": CONVERTED_DATASET_ROOT,
        "GRAPH_MODEL_CHECKPOINT": SUPPLIED_GRAPH_CHECKPOINT,
        "NODE_MODEL_CHECKPOINT": SUPPLIED_NODE_CHECKPOINT,
        "OUTPUT_ROOT": FINAL_OUTPUT_ROOT,
    },
}

stages_to_run = []
if REGENERATE_ANNOTATIONS:
    stages_to_run.append("annotation_conversion")
stages_to_run.extend(["image_resizing", "final_inference"])

prepared_notebooks: Dict[str, Path] = {}

for stage in stages_to_run:
    filename = NOTEBOOK_FILES[stage]
    source_path = NOTEBOOKS_DIR / filename
    destination_path = PREPARED_NOTEBOOKS_DIR / filename

    prepared_notebooks[stage] = prepare_notebook(
        source_path,
        destination_path,
        stage_assignments[stage],
    )
    print(f"Prepared {stage}: {destination_path}")


## 6. Execute each stage in an isolated kernel

In [ ]:
@dataclass
class StageResult:
    stage: str
    notebook: str
    started_utc: str
    finished_utc: str
    duration_seconds: float
    success: bool
    executed_notebook: str
    error: Optional[str] = None


def execute_prepared_notebook(
    stage: str,
    prepared_path: Path,
) -> StageResult:
    executed_path = EXECUTED_NOTEBOOKS_DIR / prepared_path.name
    notebook = nbformat.read(prepared_path, as_version=4)

    started = datetime.now(timezone.utc)
    start_time = time.perf_counter()
    error_message: Optional[str] = None
    success = False

    print("\n" + "=" * 80)
    print(f"RUNNING STAGE: {stage}")
    print(f"Notebook     : {prepared_path.name}")
    print("=" * 80)

    client = NotebookClient(
        notebook,
        timeout=NOTEBOOK_CELL_TIMEOUT_SECONDS,
        kernel_name=KERNEL_NAME,
        allow_errors=False,
        resources={"metadata": {"path": str(PROJECT_ROOT_PATH)}},
        record_timing=True,
    )

    try:
        client.execute()
        success = True
    except Exception as exc:
        error_message = (
            f"{type(exc).__name__}: {exc}\n\n"
            f"{traceback.format_exc()}"
        )
        print(error_message)
    finally:
        nbformat.write(notebook, executed_path)

    finished = datetime.now(timezone.utc)
    duration = time.perf_counter() - start_time

    result = StageResult(
        stage=stage,
        notebook=prepared_path.name,
        started_utc=started.isoformat(),
        finished_utc=finished.isoformat(),
        duration_seconds=duration,
        success=success,
        executed_notebook=str(executed_path),
        error=error_message,
    )

    print(
        f"Stage {'completed' if success else 'failed'} in "
        f"{duration / 60.0:.2f} minutes."
    )
    print(f"Executed notebook: {executed_path}")

    if not success and STOP_ON_FIRST_ERROR:
        raise RuntimeError(
            f"Reproduction stopped because stage '{stage}' failed. "
            f"Inspect {executed_path}."
        )

    return result


## 7. Run the reproduction

In [ ]:
stage_results: List[StageResult] = []
pipeline_started = datetime.now(timezone.utc)
pipeline_start_time = time.perf_counter()

for stage in stages_to_run:
    stage_results.append(
        execute_prepared_notebook(stage, prepared_notebooks[stage])
    )

pipeline_finished = datetime.now(timezone.utc)
pipeline_duration = time.perf_counter() - pipeline_start_time

print("\n" + "=" * 80)
print("CHECKPOINT-BASED REPRODUCTION FINISHED")
print("=" * 80)
print(f"Total duration: {pipeline_duration / 3600.0:.2f} hours")


## 8. Save the reproduction manifest

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "workflow": "checkpoint-based reproduction",
    "project_root": str(PROJECT_ROOT_PATH),
    "source_dataset": {
        "repository_url": SOURCE_DATASET_GITHUB_URL,
        "repository_path": str(SOURCE_REPOSITORY_ROOT),
        "dataset_root": str(SOURCE_DATASETS_ROOT),
        "git_commit": SOURCE_COMMIT_USED,
    },
    "regenerated_annotations": bool(REGENERATE_ANNOTATIONS),
    "stage_results": [asdict(result) for result in stage_results],
    "paths": {
        "converted_dataset_root": str(CONVERTED_DATASET_ROOT),
        "prepared_notebooks": str(PREPARED_NOTEBOOKS_DIR),
        "executed_notebooks": str(EXECUTED_NOTEBOOKS_DIR),
        "final_output": str(FINAL_OUTPUT_ROOT),
    },
    "checkpoints": {
        "graph_model": {
            "path": str(SUPPLIED_GRAPH_CHECKPOINT),
            "sha256": sha256_file(SUPPLIED_GRAPH_CHECKPOINT),
        },
        "node_model": {
            "path": str(SUPPLIED_NODE_CHECKPOINT),
            "sha256": sha256_file(SUPPLIED_NODE_CHECKPOINT),
        },
    },
    "pipeline_started_utc": pipeline_started.isoformat(),
    "pipeline_finished_utc": pipeline_finished.isoformat(),
    "pipeline_duration_seconds": pipeline_duration,
}

manifest_path = WORKSPACE_ROOT / "reproduction_manifest.json"
with manifest_path.open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

print(f"Manifest saved to : {manifest_path}")
print(f"Final outputs     : {FINAL_OUTPUT_ROOT}")
print(f"Executed notebooks: {EXECUTED_NOTEBOOKS_DIR}")
